In [1]:
  import os
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Configuration
class Config:
    # Paths
    DATA_DIR = 'pcam_dataset'
    TRAIN_X = os.path.join(DATA_DIR, 'camelyonpatch_level_2_split_train_x.h5')
    TRAIN_Y = os.path.join(DATA_DIR, 'camelyonpatch_level_2_split_train_y.h5')
    VALID_X = os.path.join(DATA_DIR, 'camelyonpatch_level_2_split_valid_x.h5')
    VALID_Y = os.path.join(DATA_DIR, 'camelyonpatch_level_2_split_valid_y.h5')
    TEST_X = os.path.join(DATA_DIR, 'camelyonpatch_level_2_split_test_x.h5')
    TEST_Y = os.path.join(DATA_DIR, 'camelyonpatch_level_2_split_test_y.h5')
    
    # Model parameters
    IMG_SIZE = 96
    BATCH_SIZE = 128
    EPOCHS = 50
    NUM_WORKERS = 4
    
    # Hyperparameters
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    DROPOUT_RATE = 0.5
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA = 0.2
    CUTMIX_ALPHA = 1.0
    CUTMIX_PROB = 0.5
    
    # Ensemble parameters
    N_FOLDS = 5
    N_TTA = 5
    
    # Directories
    MODEL_DIR = 'models'
    LOG_DIR = 'logs'
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
os.makedirs(config.MODEL_DIR, exist_ok=True)
os.makedirs(config.LOG_DIR, exist_ok=True)


# ==================== DATASET (RAM-EFFICIENT) ====================
class PCamDataset(Dataset):
    """
    PyTorch Dataset for PCam (RAM-EFFICIENT)
    Reads images from H5 file on-the-fly instead of loading all into RAM.
    """
    
    def __init__(self, x_path, y_path, transform=None, is_training=False):
        self.is_training = is_training
        self.transform = transform
        
        self.x_path = x_path
        self.y_path = y_path
        
        # This handle will be initialized in each worker process
        self.x_data = None 
        
        # Labels are small, load them into memory
        with h5py.File(self.y_path, 'r') as f:
            self.labels = f['y'][:].reshape(-1)
            
        # Get length from x_file
        with h5py.File(self.x_path, 'r') as f:
            self.length = f['x'].shape[0]
            
        print(f"Loaded {self.length} sample paths (on-disk H5 read)")
        print(f"Class distribution: {np.bincount(self.labels.astype(int))}")
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        # H5 file must be opened in __getitem__ when num_workers > 0
        if self.x_data is None:
            self.x_data = h5py.File(self.x_path, 'r')['x']
            
        # Read a single image from disk
        image = self.x_data[idx].astype(np.float32) / 255.0
        label = float(self.labels[idx])
        
        if self.transform:
            # Convert to PIL for torchvision transforms
            image = (image * 255).astype(np.uint8)
            from PIL import Image
            image = Image.fromarray(image)
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1)
            
        return image, label


def get_transforms(is_training=True):
    """Get data augmentation transforms (FIXED ORDER)"""
    
    if is_training:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            # 1. Convert to tensor FIRST
            transforms.ToTensor(),
            # 2. Normalize
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            # 3. Apply RandomErasing (cutout) to the TENSOR
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.2))
        ])
    else:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


def mixup_data(x, y, alpha=0.2):
    """Mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation"""
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    # Random box
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam


# ==================== ATTENTION MODULES ====================
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation Channel Attention"""
    
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        b, c, _, _ = x.size()
        
        # Both average and max pooling
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * out.expand_as(x)


class SpatialAttention(nn.Module):
    """Spatial Attention Module"""
    
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        
        attention = self.sigmoid(self.conv(x_cat))
        return x * attention


class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# ==================== MODEL ARCHITECTURE ====================
class DenseNetAttention(nn.Module):
    """DenseNet121 with CBAM Attention and Heavy Regularization"""
    
    def __init__(self, num_classes=1, dropout=0.5, pretrained=True):
        super(DenseNetAttention, self).__init__()
        
        # Load pretrained DenseNet121
        self.backbone = models.densenet121(pretrained=pretrained)
        
        # Remove original classifier
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        
        # Add attention module
        self.attention = CBAM(num_features, reduction=16)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Classifier with heavy dropout
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            
            nn.Linear(256, num_classes)
        )
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Extract features
        features = self.backbone.features(x)
        
        # Apply attention
        features = self.attention(features)
        
        # Global pooling
        features = self.global_pool(features)
        features = features.view(features.size(0), -1)
        
        # Classification
        output = self.classifier(features)
        
        return output


# ==================== TRAINING ====================
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, inputs, targets):
        # targets = targets.view(-1, 1)  <--- THIS LINE IS REMOVED TO FIX SHAPE MISMATCH
        
        # Apply label smoothing
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        
        return F_loss.mean()


def train_epoch(model, dataloader, criterion, optimizer, device, use_mixup=True):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device).float()
        
        # Apply MixUp or CutMix
        r = np.random.rand(1)
        if use_mixup and r < 0.5:
            if r < config.CUTMIX_PROB * 0.5:
                images, targets_a, targets_b, lam = cutmix_data(images, labels, config.CUTMIX_ALPHA)
            else:
                images, targets_a, targets_b, lam = mixup_data(images, labels, config.MIXUP_ALPHA)
            
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
        else:
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1), 'acc': 100. * correct / total})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device).float()
            
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            
            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).float()
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
            pbar.set_postfix({'loss': running_loss / (pbar.n + 1)})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = accuracy_score(all_labels, all_preds) * 100
    epoch_auc = roc_auc_score(all_labels, all_probs)
    
    return epoch_loss, epoch_acc, epoch_auc, all_labels, all_preds, all_probs


# ==================== ENSEMBLE TRAINING ====================
class EnsembleTrainer:
    """Train multiple models for ensemble"""
    
    def __init__(self, config):
        self.config = config
        self.models = []
        self.histories = []
    
    def train_single_model(self, train_loader, valid_loader, model_idx):
        """Train a single model"""
        print(f"\n{'='*60}")
        print(f"Training Model {model_idx + 1}/{config.N_FOLDS}")
        print(f"{'='*60}")
        
        # Initialize model
        model = DenseNetAttention(
            num_classes=1,
            dropout=config.DROPOUT_RATE,
            pretrained=True
        ).to(config.DEVICE)
        
        # Loss function with label smoothing
        criterion = FocalLoss(alpha=0.25, gamma=2.0, label_smoothing=config.LABEL_SMOOTHING)
        
        # Optimizer with weight decay
        optimizer = optim.AdamW(
            model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )
        
        # Learning rate schedulers
        scheduler_cosine = CosineAnnealingWarmRestarts(
            optimizer,
            T_0=10,
            T_mult=2,
            eta_min=1e-6
        )
        
        scheduler_plateau = ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=3,
            min_lr=1e-7
        )
        
        # Training loop
        best_auc = 0.0
        history = defaultdict(list)
        patience_counter = 0
        max_patience = 10
        
        for epoch in range(config.EPOCHS):
            print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
            
            # Train
            train_loss, train_acc = train_epoch(
                model, train_loader, criterion, optimizer, config.DEVICE
            )
            
            # Validate
            val_loss, val_acc, val_auc, _, _, _ = validate_epoch(
                model, valid_loader, criterion, config.DEVICE
            )
            
            # Update schedulers
            scheduler_cosine.step()
            scheduler_plateau.step(val_auc)
            
            # Log metrics
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            history['val_auc'].append(val_auc)
            history['lr'].append(optimizer.param_groups[0]['lr'])
            
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val AUC: {val_auc:.4f}")
            print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
            
            # Save best model
            if val_auc > best_auc:
                best_auc = val_auc
                patience_counter = 0
                model_path = os.path.join(config.MODEL_DIR, f'best_model_fold_{model_idx}.pth')
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_auc': val_auc,
                }, model_path)
                print(f"✓ Best model saved! AUC: {best_auc:.4f}")
            else:
                patience_counter += 1
            
            # Early stopping
            if patience_counter >= max_patience:
                print(f"\nEarly stopping triggered after {epoch+1} epochs")
                break
        
        # Load best model
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        return model, history, best_auc
    
    def train_ensemble(self, train_loader, valid_loader):
        """Train ensemble of models"""
        for i in range(config.N_FOLDS):
            model, history, best_auc = self.train_single_model(train_loader, valid_loader, i)
            self.models.append(model)
            self.histories.append(history)
            print(f"\nModel {i+1} Best AUC: {best_auc:.4f}")
        
        return self.models, self.histories


# ==================== TEST TIME AUGMENTATION (FIXED) ====================
def predict_with_tta(model, image, device, n_tta=5):
    """
    Predict with Test Time Augmentation (TENSOR-BASED)
    'image' is a single [C, H, W] normalized tensor
    """
    model.eval()
    
    # List to hold augmented images
    tta_images = [image] # 1. Original
    
    if n_tta > 1:
        # 2. Horizontal Flip
        tta_images.append(transforms.functional.hflip(image))
    if n_tta > 2:
        # 3. Vertical Flip
        tta_images.append(transforms.functional.vflip(image))
    if n_tta > 3:
        # 4. 90-degree rotation (counter-clockwise)
        # dims=(1, 2) specifies H and W dimensions
        tta_images.append(torch.rot90(image, k=1, dims=(1, 2)))
    if n_tta > 4:
        # 5. 180-degree rotation
        tta_images.append(torch.rot90(image, k=2, dims=(1, 2)))
    
    # Create a batch of all TTA images
    # torch.stack stacks a list of tensors into a new tensor
    tta_batch = torch.stack(tta_images).to(device) # [N_TTA, C, H, W]
    
    # Get predictions for the whole TTA batch at once
    with torch.no_grad():
        outputs = model(tta_batch).squeeze()
        probs = torch.sigmoid(outputs)
        
    # Average the probabilities
    return torch.mean(probs).item()


def evaluate_ensemble(models, test_loader, device, use_tta=True):
    """Evaluate ensemble with optional TTA"""
    print("\n" + "="*60)
    print("ENSEMBLE EVALUATION")
    print("="*60)
    
    all_labels = []
    all_probs = []
    
    for images, labels in tqdm(test_loader, desc='Testing'):
        batch_probs_all_models = []
        
        # 'images' is a batch [B, C, H, W]
        # 'labels' is a batch [B]
        
        for model in models:
            model.eval()
            model_batch_probs = []
            
            if use_tta:
                # Iterate over each image in the batch for TTA
                for img_tensor in images:
                    # img_tensor is [C, H, W]
                    prob = predict_with_tta(model, img_tensor, device, n_tta=config.N_TTA)
                    model_batch_probs.append(prob)
            else:
                # No TTA, predict on the whole batch
                with torch.no_grad():
                    outputs = model(images.to(device)).squeeze()
                    probs = torch.sigmoid(outputs).cpu().numpy()
                    model_batch_probs.extend(probs)
            
            # Add this model's predictions for the batch
            batch_probs_all_models.append(model_batch_probs)
        
        # Average predictions across ensemble
        # batch_probs_all_models is shape [N_Models, Batch_Size]
        # We average along axis 0 (across models)
        ensemble_probs_for_batch = np.mean(batch_probs_all_models, axis=0)
        
        all_probs.extend(ensemble_probs_for_batch)
        all_labels.extend(labels.numpy())
    
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    all_preds = (all_probs > 0.5).astype(int)
    
    # Metrics
    acc = accuracy_score(all_labels, all_preds) * 100
    auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\nEnsemble Accuracy: {acc:.2f}%")
    print(f"Ensemble AUC: {auc:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Normal', 'Tumor']))
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Tumor'],
                yticklabels=['Normal', 'Tumor'])
    plt.title(f'Ensemble Confusion Matrix (AUC: {auc:.4f})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('ensemble_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'Ensemble ROC (AUC = {auc:.4f})', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - Ensemble Model')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('ensemble_roc_curve.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    return auc, acc, all_probs


# ==================== MAIN ====================
def main():
    """Main training pipeline"""
    print("="*60)
    print("BREAST CANCER DETECTION - PyTorch with Anti-Overfitting")
    print("="*6 + "ADVANCED-CODE-FIXER")
    print(f"Device: {config.DEVICE}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    print(f"Learning Rate: {config.LEARNING_RATE}")
    print(f"Weight Decay: {config.WEIGHT_DECAY}")
    print(f"Dropout: {config.DROPOUT_RATE}")
    print(f"Label Smoothing: {config.LABEL_SMOOTHING}")
    print(f"MixUp Alpha: {config.MIXUP_ALPHA}")
    print(f"Ensemble Models: {config.N_FOLDS}")
    
    # Create datasets
    print("\nLoading datasets...")
    train_dataset = PCamDataset(
        config.TRAIN_X, config.TRAIN_Y,
        transform=get_transforms(is_training=True),
        is_training=True
    )
    
    valid_dataset = PCamDataset(
        config.VALID_X, config.VALID_Y,
        transform=get_transforms(is_training=False),
        is_training=False
    )
    
    test_dataset = PCamDataset(
        config.TEST_X, config.TEST_Y,
        transform=get_transforms(is_training=False),
        is_training=False
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    # Train ensemble
    trainer = EnsembleTrainer(config)
    models, histories = trainer.train_ensemble(train_loader, valid_loader)
    
    # Plot training history
    plot_ensemble_history(histories)
    
    # Evaluate ensemble
    auc, acc, probs = evaluate_ensemble(models, test_loader, config.DEVICE, use_tta=True)
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print(f"Final Ensemble AUC: {auc:.4f}")
    print(f"Final Ensemble Accuracy: {acc:.2f}%")
    print("="*60)
    
    return models, histories


def plot_ensemble_history(histories):
    """Plot training histories for all models"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(histories)))
    
    for idx, history in enumerate(histories):
        # Loss
        axes[0, 0].plot(history['train_loss'], label=f'Model {idx+1} Train',
                        color=colors[idx], alpha=0.7)
        axes[0, 0].plot(history['val_loss'], label=f'Model {idx+1} Val',
                        color=colors[idx], linestyle='--', alpha=0.7)
        
        # Accuracy
        axes[0, 1].plot(history['train_acc'], label=f'Model {idx+1} Train',
                        color=colors[idx], alpha=0.7)
        axes[0, 1].plot(history['val_acc'], label=f'Model {idx+1} Val',
                        color=colors[idx], linestyle='--', alpha=0.7)
        
        # AUC
        axes[1, 0].plot(history['val_auc'], label=f'Model {idx+1}',
                        color=colors[idx], alpha=0.7)
        
        # Learning Rate
        axes[1, 1].plot(history['lr'], label=f'Model {idx+1}',
                        color=colors[idx], alpha=0.7)
    
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_title('Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_title('Validation AUC')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('AUC')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('ensemble_training_history.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("\n✓ Training history plot saved!")


if __name__ == "__main__":
    # Check CUDA availability
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("WARNING: CUDA not available, using CPU")
    
    # Run training
    models, histories = main()

GPU: NVIDIA L40S
GPU Memory: 47.67 GB
BREAST CANCER DETECTION - PyTorch with Anti-Overfitting
======ADVANCED-CODE-FIXER
Device: cuda
Batch Size: 128
Learning Rate: 0.0003
Weight Decay: 0.0001
Dropout: 0.5
Label Smoothing: 0.1
MixUp Alpha: 0.2
Ensemble Models: 5

Loading datasets...
Loaded 262144 sample paths (on-disk H5 read)
Class distribution: [131072 131072]
Loaded 32768 sample paths (on-disk H5 read)
Class distribution: [16399 16369]
Loaded 32768 sample paths (on-disk H5 read)
Class distribution: [16391 16377]

Training Model 1/5

Epoch 1/50


Validation: 100%|██████████| 256/256 [00:08<00:00, 31.05it/s, loss=0.0217]


Train Loss: 0.0321 | Train Acc: 76.57%
Val Loss: 0.0217 | Val Acc: 86.97% | Val AUC: 0.9461
Learning Rate: 2.93e-04
✓ Best model saved! AUC: 0.9461

Epoch 2/50


Validation: 100%|██████████| 256/256 [00:08<00:00, 31.14it/s, loss=0.0199]


Train Loss: 0.0260 | Train Acc: 80.43%
Val Loss: 0.0199 | Val Acc: 89.07% | Val AUC: 0.9572
Learning Rate: 2.71e-04
✓ Best model saved! AUC: 0.9572

Epoch 3/50


...

Train Loss: 0.0161 | Train Acc: 86.85%
Val Loss: 0.0248 | Val Acc: 88.35% | Val AUC: 0.9622
Learning Rate: 1.04e-04

Epoch 23/50


Validation: 100%|██████████| 256/256 [00:07<00:00, 32.21it/s, loss=0.0274]


Train Loss: 0.0157 | Train Acc: 87.42%
Val Loss: 0.0273 | Val Acc: 88.72% | Val AUC: 0.9605
Learning Rate: 8.26e-05

Early stopping triggered after 23 epochs

Model 5 Best AUC: 0.9707

✓ Training history plot saved!

ENSEMBLE EVALUATION


Testing: 100%|██████████| 256/256 [19:06<00:00,  4.48s/it]



Ensemble Accuracy: 86.84%
Ensemble AUC: 0.9652

Classification Report:
              precision    recall  f1-score   support

      Normal       0.80      0.98      0.88     16391
       Tumor       0.98      0.75      0.85     16377

    accuracy                           0.87     32768
   macro avg       0.89      0.87      0.87     32768
weighted avg       0.89      0.87      0.87     32768


TRAINING COMPLETE!
Final Ensemble AUC: 0.9652
Final Ensemble Accuracy: 86.84%


In [1]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

# ============================================================
# CONFIGURATION
# ============================================================
class Config:
    DATA_DIR = "pcam_dataset"

    TEST_X = os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_x.h5")
    TEST_Y = os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_y.h5")

    SAVE_DIR = "opdense121att"
    os.makedirs(SAVE_DIR, exist_ok=True)

    BATCH_SIZE = 128
    NUM_WORKERS = 2
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    BEST_MODEL_PATH = "experiments/best_model_fold_4-Copy1.pth"


config = Config()

# ============================================================
# DATASET
# ============================================================
class PCamDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None):
        self.x_path = x_path
        self.y_path = y_path
        self.transform = transform

        self.x_file = None
        self.x_data = None

        with h5py.File(y_path, "r") as f:
            self.labels = f["y"][:].reshape(-1)

        with h5py.File(x_path, "r") as f:
            self.length = f["x"].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.x_file is None:
            self.x_file = h5py.File(self.x_path, "r")
            self.x_data = self.x_file["x"]

        img = self.x_data[idx].astype("float32") / 255.0
        label = float(self.labels[idx])

        if self.transform:
            img = (img * 255).astype(np.uint8)
            from PIL import Image
            img = Image.fromarray(img)
            img = self.transform(img)
        else:
            img = torch.tensor(img).permute(2, 0, 1)

        return img, label


def get_transforms(is_training=False):
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])


# ============================================================
# CBAM ATTENTION MODULES (MATCH EXACT TRAINING)
# ============================================================
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b,c))
        max_out = self.fc(self.max_pool(x).view(b,c))
        out = self.sigmoid(avg_out + max_out).view(b,c,1,1)
        return x * out


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2,1,kernel_size,padding=kernel_size//2,bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg, max_], dim=1)
        att = self.sigmoid(self.conv(out))
        return x * att


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# ============================================================
# FINAL MODEL (EXACT MATCH TO TRAINING)
# ============================================================
class DenseNetAttention(nn.Module):
    """DenseNet121 with CBAM (same as training checkpoint)"""
    def __init__(self, num_classes=1, dropout=0.5, pretrained=False):
        super(DenseNetAttention, self).__init__()

        self.backbone = models.densenet121(weights=None)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()

        # THE NAME MUST MATCH TRAINING: "attention"
        self.attention = CBAM(num_features, reduction=16)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(num_features,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout*0.5),

            nn.Linear(256,1)
        )

    def forward(self,x):
        f = self.backbone.features(x)
        f = self.attention(f)
        f = self.pool(f)
        f = f.view(f.size(0), -1)
        return self.classifier(f)


# ============================================================
# LOAD MODEL CHECKPOINT
# ============================================================
print("\nLoading model:", config.BEST_MODEL_PATH)

model = DenseNetAttention()
checkpoint = torch.load(config.BEST_MODEL_PATH, map_location=config.DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(config.DEVICE)
model.eval()

print("✓ Model loaded successfully!\n")


# ============================================================
# LOAD TEST LOADER
# ============================================================
test_dataset = PCamDataset(
    config.TEST_X,
    config.TEST_Y,
    transform=get_transforms(False)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print("✓ Test Loader Ready. Samples =", len(test_dataset))


# ============================================================
# INFERENCE
# ============================================================
all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(config.DEVICE)

        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print("\n✓ Inference Complete!")


# ============================================================
# THRESHOLD OPTIMIZATION (MAX AUC + MAX ACC)
# ============================================================
thresholds = np.linspace(0.05, 0.95, 181)
best_thresh = 0.5
best_score = -1

base_auc = roc_auc_score(all_labels, all_probs)

for th in thresholds:
    preds = (all_probs > th).astype(int)
    acc = accuracy_score(all_labels, preds)
    score = acc + base_auc  # maximize both

    if score > best_score:
        best_score = score
        best_thresh = th

print(f"\n🔥 OPTIMIZED THRESHOLD = {best_thresh:.4f}\n")


# ============================================================
# FINAL PREDICTIONS
# ============================================================
final_preds = (all_probs > best_thresh).astype(int)


# ============================================================
# METRICS
# ============================================================
acc = accuracy_score(all_labels, final_preds)
prec = precision_score(all_labels, final_preds)
rec = recall_score(all_labels, final_preds)
f1 = f1_score(all_labels, final_preds)
auc = roc_auc_score(all_labels, all_probs)

cm = confusion_matrix(all_labels, final_preds)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)
print("AUC      :", auc)


# ============================================================
# SAVE RESULTS
# ============================================================
report_path = os.path.join(config.SAVE_DIR, "metrics_report.txt")

with open(report_path, "w") as f:
    f.write("=== FINAL TEST RESULTS ===\n")
    f.write(f"Optimized Threshold: {best_thresh}\n")
    f.write(f"Accuracy:  {acc:.4f}\n")
    f.write(f"Precision: {prec:.4f}\n")
    f.write(f"Recall:    {rec:.4f}\n")
    f.write(f"F1 Score:  {f1:.4f}\n")
    f.write(f"AUC:       {auc:.4f}\n")
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))

print("✓ metrics_report.txt saved")


# ============================================================
# SAVE CONFUSION MATRIX
# ============================================================
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal","Tumor"],
            yticklabels=["Normal","Tumor"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(config.SAVE_DIR, "confusion_matrix.png"), dpi=300)
plt.close()

print("✓ Saved: confusion_matrix.png")


# ============================================================
# SAVE ROC CURVE
# ============================================================
fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC={auc:.4f}")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(config.SAVE_DIR, "roc_curve.png"), dpi=300)
plt.close()

print("✓ Saved: roc_curve.png")


# ============================================================
# SAVE PRECISION-RECALL CURVE
# ============================================================
prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probs)
plt.figure(figsize=(6,5))
plt.plot(rec_curve, prec_curve)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.savefig(os.path.join(config.SAVE_DIR, "pr_curve.png"), dpi=300)
plt.close()

print("✓ Saved: pr_curve.png")

print("\n🎉 All evaluation results saved in:", config.SAVE_DIR)



Loading model: experiments/best_model_fold_4-Copy1.pth


/home/jupyter-238w1a0478/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


✓ Model loaded successfully!

✓ Test Loader Ready. Samples = 32768

✓ Inference Complete!

🔥 OPTIMIZED THRESHOLD = 0.3050

Accuracy : 0.89324951171875
Precision: 0.899002416506599
Recall   : 0.8859375954081944
F1 Score : 0.8924221921515562
AUC      : 0.9610433749523959
✓ metrics_report.txt saved
✓ Saved: confusion_matrix.png
✓ Saved: roc_curve.png
✓ Saved: pr_curve.png

🎉 All evaluation results saved in: opdense121att
